# CUDA Fundamentals

Establish the CUDA execution concepts needed for later GPU experiments.

## Objectives

Detect CUDA safely and frame an experiment around host/device execution and synchronization.

## Background

CUDA launches work on a GPU execution hierarchy; asynchronous execution means correct timing requires deliberate synchronization.

## Prediction

A CUDA operation has at least three distinguishable costs:

1. host-side launch overhead;
2. GPU execution time;
3. host-side waiting caused by synchronization.

Because CUDA launches are normally asynchronous, measuring only the Python call duration should initially report mostly host-side enqueue overhead rather than completed GPU work.

The first CUDA operation may be slower than later operations because the CUDA context, memory allocator, and kernel machinery may require one-time initialization.

For very small tensors, fixed launch and synchronization overhead should dominate. As tensor size increases, GPU execution time should become a larger fraction of total elapsed time.

## Environment

In [1]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/00-foundations


## Experiment

In [4]:
from pprint import pprint

from common.cuda import detect_cuda


cuda_info = detect_cuda()
pprint(cuda_info)

if not cuda_info.torch_installed:
    raise RuntimeError(
        "PyTorch is not installed in this environment. "
        "CUDA experiments cannot continue."
    )

if not cuda_info.available:
    raise RuntimeError(
        "PyTorch is installed, but CUDA is not available. "
        f"Detection error: {cuda_info.error!r}"
    )

CudaInfo(torch_installed=True,
         available=True,
         device_count=1,
         device_names=('NVIDIA GB10',),
         torch_version='2.13.0+cu130',
         cuda_version='13.0',
         error=None)


### CUDA runtime and device properties

Before measuring execution, inspect the CUDA runtime exposed through PyTorch and the properties of the selected device.

These values are environment facts. They describe the software and hardware visible to this process but do not yet measure performance.

In [5]:
import torch


device_index = torch.cuda.current_device()
device = torch.device(f"cuda:{device_index}")
properties = torch.cuda.get_device_properties(device_index)

runtime_info = {
    "torch_version": torch.__version__,
    "torch_cuda_build_version": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "device_count": torch.cuda.device_count(),
    "current_device_index": device_index,
    "current_device_name": torch.cuda.get_device_name(device_index),
    "compute_capability": (
        properties.major,
        properties.minor,
    ),
    "multiprocessor_count": properties.multi_processor_count,
    "total_device_memory_bytes": properties.total_memory,
    "total_device_memory_gib": properties.total_memory / 1024**3,
}

pprint(runtime_info)

{'compute_capability': (12, 1),
 'cuda_available': True,
 'current_device_index': 0,
 'current_device_name': 'NVIDIA GB10',
 'device_count': 1,
 'multiprocessor_count': 48,
 'torch_cuda_build_version': '13.0',
 'torch_version': '2.13.0+cu130',
 'total_device_memory_bytes': 130663002112,
 'total_device_memory_gib': 121.68940353393555}


In [6]:
values = torch.arange(
    8,
    dtype=torch.float32,
    device=device,
)

result = values * 2.0 + 1.0

print(f"values device: {values.device}")
print(f"result device: {result.device}")
print(f"result: {result.cpu().tolist()}")

expected = [1.0, 3.0, 5.0, 7.0, 9.0, 11.0, 13.0, 15.0]
assert result.cpu().tolist() == expected

values device: cuda:0
result device: cuda:0
result: [1.0, 3.0, 5.0, 7.0, 9.0, 11.0, 13.0, 15.0]


## Observations

TODO: Record detection output and measurements only from the current environment.

## Explanation

TODO: Explain launch, execution, and synchronization costs separately.

## Connection to LLMs

LLM kernels rely on CUDA's execution hierarchy, asynchronous launches, and efficient batches of parallel work.

## Further Exploration

TODO: Compare cold-start, warmed-up, synchronized, and unsynchronized timing.